# Extract the list of trainings

In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
from pathlib import Path
import pandas as pd

# set the folder
folder = Path('/run/user/1000/gvfs/smb-share:server=istore,share=microscopy_group_share,user=jeromeb/Admin/Usage Stats/2024')
booking_types = ['default','service', 'maintenance', 'training']
divisions = ['Unknown','Cell Biology','Neurobiology','PNAS','Structural Studies','University']
groups = pd.read_csv(folder/'groups.csv')
users = pd.read_csv(folder/'users.csv')
calendars = pd.read_csv(folder/'calendars.csv')
# get a cookie from the curl command copied from firefox dev tools
with open('cookie.txt','r') as f:
    curl_str = f.read()
db_file = 'training.db'

In [30]:
from usagestats.bookingdb import BookingDB

with BookingDB(db_file, curl_str=curl_str) as db:
    db.create_calendars_from_df(calendars)
    df = db.create_training_request()
    

Restrict the booking for a specific time

In [49]:
from datetime import datetime, timedelta
import pytz
local_tz = pytz.timezone("Europe/London")
dfs = df[df['start']>datetime.fromisoformat("2024-01-01").replace(tzinfo=local_tz)]

Sort by name and data

In [51]:
dfs = dfs.sort_values(["email","start"])

Export to a csv file

In [ ]:
dfs[['user','instrument','start','attendees','subject']].to_csv(folder/'training.csv')